# Fairness Evaluation Notebook

**Purpose:** Measure model reliability across diverse image conditions and optional metadata groups.

**Goal:** Identify reliability gaps, not profile users.

**Rule:** Use only de-identified, consent-approved, aggregate data. No patient identifiers in outputs.

Follows: `docs/product/12_RESEARCH_FAIRNESS_MONITORING_HANDHOLDING.md` Step 3.

---

## Sections
1. Setup and data loading
2. Class balance and dataset overview
3. Performance by skin tone category (optional — requires metadata consent)
4. Performance by body region
5. Performance by image quality band
6. Performance by lighting condition
7. Performance by device / camera type
8. Calibration by group
9. Summary and gap report

In [ ]:
# 1. Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    confusion_matrix, classification_report
)

sns.set_theme(style='whitegrid')
print('Setup complete')

In [ ]:
# 2. Data loading
# Replace this path with your de-identified evaluation export.
# Required columns:
#   - prediction_id (str)
#   - true_label (int: 0=benign, 1=malignant)
#   - predicted_label (int)
#   - calibrated_confidence (float, 0-1)
#   - model_version (str)
#   - image_quality (str: 'ok' | 'blur' | 'dark' | 'glare')
#   - body_region (str, optional)
#   - skin_tone_category (str, optional — only if consent + ethics allow)
#   - lighting_condition (str, optional)
#   - camera_type (str, optional)

DATA_PATH = '../outputs/eval_export.csv'  # TODO: replace with real path

try:
    df = pd.read_csv(DATA_PATH)
    print(f'Loaded {len(df)} evaluation records')
    print(df.dtypes)
except FileNotFoundError:
    print(f'Evaluation export not found at {DATA_PATH}.')
    print('Run the evaluation pipeline first or point DATA_PATH to your export.')
    df = pd.DataFrame()  # empty frame so cells below show structure

In [ ]:
# 3. Dataset overview
if df.empty:
    print('No data loaded — skipping.')
else:
    print('Class balance:')
    print(df['true_label'].value_counts())

    overall_acc = accuracy_score(df['true_label'], df['predicted_label'])
    print(f'\nOverall accuracy: {overall_acc:.4f}')
    print(classification_report(df['true_label'], df['predicted_label'],
                                target_names=['Benign', 'Malignant']))

In [ ]:
# Helper: compute per-group metrics
def group_metrics(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    """Return accuracy, sensitivity, specificity, AUC per group."""
    if group_col not in df.columns or df.empty:
        print(f'Column {group_col!r} not available — skipping.')
        return pd.DataFrame()
    rows = []
    for group, gdf in df.groupby(group_col):
        if len(gdf) < 5:
            continue  # skip tiny groups
        cm = confusion_matrix(gdf['true_label'], gdf['predicted_label'], labels=[0, 1])
        tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else None
        specificity = tn / (tn + fp) if (tn + fp) > 0 else None
        try:
            auc = roc_auc_score(gdf['true_label'], gdf['calibrated_confidence'])
        except Exception:
            auc = None
        rows.append({
            group_col: group,
            'n': len(gdf),
            'accuracy': accuracy_score(gdf['true_label'], gdf['predicted_label']),
            'sensitivity': sensitivity,
            'specificity': specificity,
            'auc': auc,
        })
    return pd.DataFrame(rows).set_index(group_col)

In [ ]:
# 4. Performance by image quality band
quality_metrics = group_metrics(df, 'image_quality')
if not quality_metrics.empty:
    display(quality_metrics.round(4))
    quality_metrics['accuracy'].plot(kind='bar', title='Accuracy by Image Quality')
    plt.tight_layout()
    plt.show()

In [ ]:
# 5. Performance by body region
body_metrics = group_metrics(df, 'body_region')
if not body_metrics.empty:
    display(body_metrics.round(4))
    body_metrics['accuracy'].plot(kind='bar', title='Accuracy by Body Region', figsize=(10, 4))
    plt.tight_layout()
    plt.show()

In [ ]:
# 6. Performance by skin tone category
# IMPORTANT: Only run this cell if:
#   (a) skin_tone_category metadata was collected with patient consent
#   (b) ethical review approved this analysis
#   (c) data is fully de-identified
skin_tone_metrics = group_metrics(df, 'skin_tone_category')
if not skin_tone_metrics.empty:
    display(skin_tone_metrics.round(4))
    skin_tone_metrics['accuracy'].plot(kind='bar', title='Accuracy by Skin Tone Category')
    plt.tight_layout()
    plt.show()

In [ ]:
# 7. Performance by lighting condition
lighting_metrics = group_metrics(df, 'lighting_condition')
if not lighting_metrics.empty:
    display(lighting_metrics.round(4))

In [ ]:
# 8. Performance by camera / device type
camera_metrics = group_metrics(df, 'camera_type')
if not camera_metrics.empty:
    display(camera_metrics.round(4))

In [ ]:
# 9. Calibration by group — reliability diagram
if not df.empty and 'calibrated_confidence' in df.columns:
    from sklearn.calibration import calibration_curve
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    prob_true, prob_pred = calibration_curve(
        df['true_label'], df['calibrated_confidence'], n_bins=10
    )
    ax.plot(prob_pred, prob_true, marker='o', label='Model')
    ax.set_xlabel('Mean predicted confidence')
    ax.set_ylabel('Fraction of positives')
    ax.set_title('Reliability diagram (overall)')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# 10. Gap report — flag groups with accuracy gap > threshold
GAP_THRESHOLD = 0.05  # flag groups with accuracy > 5pp below overall

if not df.empty:
    overall_acc = accuracy_score(df['true_label'], df['predicted_label'])
    print(f'Overall accuracy: {overall_acc:.4f}\n')
    print(f'Groups with accuracy gap > {GAP_THRESHOLD:.0%}:')
    for name, gm in [('Image quality', quality_metrics),
                     ('Body region', body_metrics),
                     ('Skin tone', skin_tone_metrics)]:
        if gm.empty:
            continue
        flagged = gm[gm['accuracy'] < overall_acc - GAP_THRESHOLD]
        if not flagged.empty:
            print(f'\n{name}:')
            display(flagged[['n', 'accuracy']].round(4))
    print('\nDone. Review flagged groups for model improvement opportunities.')